# Macroeconomic Stress Testing for Retail Loan Defaults
 
**Core Research Question:** Do macroeconomic indicators (unemployment rate, inflation CPI, Fed funds rate) significantly improve retail loan default prediction compared to traditional models relying strictly on individual borrower credit profiles (FICO, DTI, Income)?

---
### End-to-End Quantitative Pipeline Overview
1. **Phase 1:** Environment Setup & Toolchain Initialization
2. **Phase 2:** Data Procurement (LendingClub Micro-data & FRED Macro Series)
3. **Phase 3:** Data Cleaning & Point-in-Time (PIT) Chronological Merging (`pd.merge_asof`)
4. **Phase 4:** Feature Engineering & Preprocessing (Macro-Borrower Interaction Terms)
5. **Phase 5:** Model Training & Out-of-Time (OOT) Validation (Chronological Splits)
6. **Phase 6:** Evaluation Metrics & Probability Calibration (PR-AUC, ROC-AUC, Brier Score)
7. **Phase 7:** Model Coefficient Export for Static GitHub Pages Interactive Application

## Phase 1: Environment Setup & Toolchain Initialization

In [2]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, 
    precision_recall_curve, 
    auc, 
    brier_score_loss, 
    classification_report, 
    confusion_matrix
)
from sklearn.calibration import calibration_curve

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 50)
print('All primary libraries imported successfully.')

All primary libraries imported successfully.


## Phase 2: Data Procurement (Raw Materials)
- **Borrower Micro-Data:** LendingClub Loan Dataset (origination dates `issue_d`, FICO, DTI, Income, target `loan_status`)
- **Macroeconomic Series:** FRED API (`UNRATE`, `CPIAUCSL`, `FEDFUNDS`)

In [4]:
# Phase 2: Load Curated Datasets
# 1. Micro-Data (LendingClub filtered to 15 relevant features)
microdata_path = 'data/processed/microdataprocessed.csv'
print(f'Loading micro-data from {microdata_path}...')
microdata = pd.read_csv(microdata_path, low_memory=False)
print(f'Micro-data loaded: {microdata.shape[0]:,} rows, {microdata.shape[1]} columns.')

# 2. Macroeconomic Series (FRED monthly indicators 2007-2018)
macrodata_path = 'data/processed/macrodata.csv'
print(f'Loading macro-data from {macrodata_path}...')
macrodata = pd.read_csv(macrodata_path)
print(f'Macro-data loaded: {macrodata.shape[0]} months, {macrodata.shape[1]} columns.')


Loading micro-data from data/processed/microdataprocessed.csv...
Micro-data loaded: 2,260,701 rows, 15 columns.
Loading macro-data from data/processed/macrodata.csv...
Macro-data loaded: 144 months, 7 columns.


In [5]:
# Display previews of loaded datasets
print('=== LendingClub Micro-Data Preview ===')
display(microdata.head())

print('=== FRED Macroeconomic Data Preview ===')
display(macrodata.head())


=== LendingClub Micro-Data Preview ===


,loan_amnt,term,int_rate,installment,home_ownership,annual_inc,issue_d,loan_status,dti,fico_range_low,fico_range_high,open_acc,pub_rec,revol_util,mort_acc
0,3600.0,36 months,13.99,123.03,MORTGAGE,55000.0,Dec-2015,Fully Paid,5.91,675.0,679.0,7.0,0.0,29.7,1.0
1,24700.0,36 months,11.99,820.28,MORTGAGE,65000.0,Dec-2015,Fully Paid,16.06,715.0,719.0,22.0,0.0,19.2,4.0
2,20000.0,60 months,10.78,432.66,MORTGAGE,63000.0,Dec-2015,Fully Paid,10.78,695.0,699.0,6.0,0.0,56.2,5.0
3,35000.0,60 months,14.85,829.90,MORTGAGE,110000.0,Dec-2015,Current,17.06,785.0,789.0,13.0,0.0,11.6,1.0
4,10400.0,60 months,22.45,289.91,MORTGAGE,104433.0,Dec-2015,Fully Paid,25.37,695.0,699.0,12.0,0.0,64.5,6.0


=== FRED Macroeconomic Data Preview ===


,date,UNRATE,CPIAUCSL,FEDFUNDS,UMCSENT,DGS10,CPI_YoY_Inflation
0,2007-01-01,4.6,203.437,5.25,96.9,4.76,2.08
1,2007-02-01,4.5,204.226,5.26,91.3,4.72,2.42
2,2007-03-01,4.4,205.288,5.26,88.4,4.56,2.80
3,2007-04-01,4.5,205.904,5.25,87.1,4.69,2.59
4,2007-05-01,4.4,206.755,5.25,88.3,4.75,2.71


## Phase 3: Data Cleaning & Point-in-Time (PIT) Chronological Merging
Key integrity requirements:
1. **Target Filtering:** Retain only completed outcomes: `Charged Off` (1) and `Fully Paid` (0). Exclude ongoing `Current` loans to prevent right-censoring bias.
2. **Eliminate Future Leakage:** Drop post-origination columns (`recoveries`, `total_pymnt`, `last_pymnt_d`, etc.) — already secured via the 15-feature application-time allowlist.
3. **Publication Lag:** Shift macroeconomic indicators by 1 month (`macro_df[indicators].shift(1)`) to reflect realistic economic reporting lags.
4. **Point-in-Time Merge:** Join borrower micro-data with prevailing lagged macro indicators based on loan origination month (`issue_d`).

In [ ]:
# Phase 3: Data Cleaning & Point-in-Time (PIT) Chronological Merging

# 1. Clean missing origination dates, filter completed outcomes, and convert issue_d to datetime
print('Preparing micro-data dates and filtering terminal outcomes...')
micro_df = microdata.dropna(subset=['issue_d']).copy()

# Filter completed loan outcomes to prevent target censoring (Fully Paid vs Charged Off)
terminal_statuses = ['Fully Paid', 'Charged Off']
micro_df = micro_df[micro_df['loan_status'].isin(terminal_statuses)].copy()

# Encode binary target: 1 for Charged Off (Default), 0 for Fully Paid
micro_df['default'] = (micro_df['loan_status'] == 'Charged Off').astype(int)

# Standardize origination date to monthly datetime (start of month)
micro_df['issue_date'] = pd.to_datetime(micro_df['issue_d'], format='%b-%Y')
micro_df = micro_df.sort_values('issue_date').reset_index(drop=True)

# 2. Prepare macro-data with 1-month publication lag
print('Applying 1-month publication lag to macro indicators...')
macro_df = macrodata.copy()
macro_df['macro_date'] = pd.to_datetime(macro_df['date'])
macro_df = macro_df.sort_values('macro_date').reset_index(drop=True)

macro_indicators = ['UNRATE', 'CPIAUCSL', 'FEDFUNDS', 'UMCSENT', 'DGS10', 'CPI_YoY_Inflation']
macro_df[macro_indicators] = macro_df[macro_indicators].shift(1)

# 3. Point-in-Time (PIT) Merge: Match each loan to prevailing macro climate
print('Merging micro-data and macro-data into unified dataframe df...')
df = pd.merge(
    micro_df,
    macro_df.drop(columns=['date']),
    left_on='issue_date',
    right_on='macro_date',
    how='inner'
).drop(columns=['macro_date'])

# Drop any edge records with missing macro data resulting from the 1-month lag
df = df.dropna(subset=macro_indicators).reset_index(drop=True)

print(f'Successfully created unified dataframe df: {df.shape[0]:,} rows, {df.shape[1]} columns.')


In [ ]:
# Display preview and summary diagnostics of unified dataset
print('=== Unified Micro + Macro Dataset Preview (First 5 Records) ===')
display(df.head())

print('\n=== Dataset Diagnostics & Integrity Checks ===')
print(f'Total observations: {len(df):,}')
print(f'Origination date span: {df["issue_date"].min().strftime("%b %Y")} to {df["issue_date"].max().strftime("%b %Y")}')
print(f'Target distribution: {df["default"].value_counts().to_dict()} (Default Rate: {df["default"].mean():.2%})')
print(f'Missing values across macro indicators: {df[macro_indicators].isna().sum().to_dict()}')


## Phase 4: Feature Engineering & Preprocessing
- Construct Macro-Borrower Interaction Terms:
  - `dti_x_inflation = DTI * CPI_rate`
  - `income_to_unemp = Annual_Income / Unemployment_Rate`
- Median Imputation for missing values
- Standard scaling fitted strictly on training data

In [ ]:
# TODO: Feature engineering and strictly partitioned scaling

## Phase 5: Model Training & Out-of-Time (OOT) Validation
- Chronological Split: Train on 2013–2016 loans, Test/Validate on 2017 loans (Zero future look-ahead bias).
- Econometric Baseline: `statsmodels.api.Logit` to determine p-values and logit odds ratios.
- Machine Learning Model: `scikit-learn` `LogisticRegression(class_weight='balanced')`.

In [ ]:
# TODO: Chronological split and model fitting

## Phase 6: Evaluation Metrics & Probability Calibration
- PR-AUC (Precision-Recall Area Under Curve) as primary metric for class imbalance.
- ROC-AUC and Brier Score Loss $BS = \frac{1}{N} \sum (f_i - y_i)^2$.
- Probability Calibration Curves (Reliability diagrams).

In [ ]:
# TODO: Calculate evaluation metrics and plot calibration curves

## Phase 7: ES 335 Deliverable — Static Web Model Export
- Extract model coefficients (`model.coef_`) and intercept (`model.intercept_`).
- Export to `model_coefficients.json` for client-side JavaScript probability calculation:  
  $$P = \frac{1}{1 + e^{-(\beta_0 + \sum \beta_i X_i)}}$$

In [ ]:
# TODO: Export weights to model_coefficients.json for GitHub Pages web app